In [1]:
import pandas as pd
from pathlib import Path


In [ ]:
data_path = Path("data/raw")   

In [3]:
orders = pd.read_csv(data_path / "olist_orders_dataset.csv")
order_items = pd.read_csv(data_path / "olist_order_items_dataset.csv")
customers = pd.read_csv(data_path / "olist_customers_dataset.csv")
products = pd.read_csv(data_path / "olist_products_dataset.csv")
sellers = pd.read_csv(data_path / "olist_sellers_dataset.csv")
payments = pd.read_csv(data_path / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(data_path / "olist_order_reviews_dataset.csv")

In [4]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


Orders

Grain: One row per order
Primary Key: order_id
Foreign Key: customer_id
Purpose: Captures order lifecycle and delivery information.
    
Order Items

Grain: One row per order line
Composite Key: order_id + order_item_id
Foreign Keys: product_id, seller_id
Purpose: Captures products sold within each order.

# 1. Let's, understand the data

1.How many rows are in each file
2.How many columns
3.Column names
4.Missing values
5.Data types


In [ ]:
import pandas as pd
from pathlib import Path

data_path = Path("data/raw")

for file in sorted(data_path.glob("*.csv")):
    df = pd.read_csv(file, nrows=5)

    print("\n" + "=" * 60)
    print(file.name)
    print("Columns:", len(df.columns))
    print(list(df.columns))


olist_customers_dataset.csv
Columns: 5
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

olist_geolocation_dataset.csv
Columns: 5
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

olist_order_items_dataset.csv
Columns: 7
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

olist_order_payments_dataset.csv
Columns: 5
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

olist_order_reviews_dataset.csv
Columns: 7
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

olist_orders_dataset.csv
Columns: 8
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

olist

- Orders — central transaction table - > olist_orders_dataset
This tells us when an order happened and its lifecycle.

- Order Items — sales detail
    olist_order_items_dataset
    It tells us: what product → sold in which order → by which seller → at what price → with freight cost
    
- Customers
    olist_customers_dataset: This gives the customer attributes.

- Products
    olist_products_dataset
    
- Sellers
    olist_sellers_dataset
    
- Reviews
    olist_order_reviews_dataset
    This is particularly interesting because it lets us connect:Sales + Delivery + Customer satisfaction

#  2. Data model           
                 
                 dim_customer
                         │
                         │
                         ▼
                    fact_orders
                         │
              ┌──────────┴──────────┐
              │                     │
              ▼                     ▼
        fact_order_items       fact_payments
              │
        ┌─────┴─────┐
        ▼           ▼
   dim_product   dim_seller
        │
        ▼
 dim_category

# 3. Business problem

The objective of this analysis is to understand overall sales performance, customer purchasing behavior, and delivery efficiency. The analysis focuses on the following key business questions:

- Revenue Performance: How much revenue is being generated, and how does revenue change over time?
- Sales Trends: What are the overall sales trends across different time periods?
- Product & Category Performance: Which products and product categories contribute the most to total revenue?
- Average Order Value: What is the average value of a customer order?
- Delivery Performance: How long does it typically take to deliver an order to the customer?
- Late Deliveries: What percentage of orders are delivered later than the estimated delivery date?

# 4. Data Limitation

The dataset does not contain an explicit product cost or profit field. Although it includes price and freight_value, these values do not represent the actual cost incurred by the business.

Therefore, gross profit, profit margin, or profitability cannot be accurately calculated from this dataset.

Instead, the analysis will focus on metrics that can be reliably derived from the available data, including:

Revenue
Order Value
Item Quantity
Freight Cost
Average Order Value (AOV)
Delivery Time
Late Delivery Rate
Review Score
Seller Performance
Customer Repeat Rate


# 5. Relationships and data quality.

Lets check row counts and key uniqueness,

In [ ]:
import pandas as pd
from pathlib import Path

data_path = Path("data/raw")

files = [
    "olist_customers_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
]

for filename in files:
    file = data_path / filename
    df = pd.read_csv(file)

    print(f"{filename}: {len(df):,} rows")


olist_customers_dataset.csv: 99,441 rows
olist_order_items_dataset.csv: 112,650 rows
olist_order_payments_dataset.csv: 103,886 rows
olist_order_reviews_dataset.csv: 99,224 rows
olist_orders_dataset.csv: 99,441 rows
olist_products_dataset.csv: 32,951 rows
olist_sellers_dataset.csv: 3,095 rows


| Table       |    Rows | Likely grain                          |
| ----------- | ------: | ------------------------------------- |
| Customers   |  99,441 | 1 row per customer/order relationship |
| Orders      |  99,441 | 1 row per order                       |
| Order Items | 112,650 | 1 row per product within an order     |
| Payments    | 103,886 | 1 row per payment transaction         |
| Reviews     |  99,224 | Review records                        |
| Products    |  32,951 | 1 row per product                     |
| Sellers     |   3,095 | 1 row per seller                      |


Before building the data model, it is important to understand the grain of each table — in other words, what a single row represents.

Understanding table grain is fundamental to accurate BI analysis and data modeling, as it determines how tables should be joined and how metrics should be calculated.

For example:

- Orders: One row represents one customer order.
- Order Items: One row represents one product line within an order.
- Customers: One row represents one customer.
- Products: One row represents one product.
- Sellers: One row represents one seller.
- Payments: One row represents one payment transaction associated with an order.
- Reviews: One row represents one order review.

This explains why the number of order-item records is greater than the number of orders:

112,650 order-item rows > 99,441 orders

A single customer can place multiple orders, and a single order can contain multiple products.

Understanding these relationships is essential for preventing double-counting, defining accurate KPIs, and designing an effective dimensional data model.

# 6. Let's identify the keys


In [ ]:
#python3 - <<'PY'
import pandas as pd
from pathlib import Path

data_path = Path("data/raw")

checks = {
    "olist_customers_dataset.csv": ["customer_id", "customer_unique_id"],
    "olist_orders_dataset.csv": ["order_id", "customer_id"],
    "olist_products_dataset.csv": ["product_id"],
    "olist_sellers_dataset.csv": ["seller_id"],
    "olist_order_items_dataset.csv": ["order_id", "order_item_id", "product_id", "seller_id"],
    "olist_order_payments_dataset.csv": ["order_id", "payment_sequential"],
    "olist_order_reviews_dataset.csv": ["review_id", "order_id"],
}

for filename, columns in checks.items():
    df = pd.read_csv(data_path / filename)

    print("\n" + "=" * 70)
    print(filename)

    for col in columns:
        unique_count = df[col].nunique(dropna=False)
        row_count = len(df)
        duplicates = row_count - unique_count

        print(f"{col:30} Unique: {unique_count:,} | Duplicates: {duplicates:,}")
#PY


olist_customers_dataset.csv
customer_id                    Unique: 99,441 | Duplicates: 0
customer_unique_id             Unique: 96,096 | Duplicates: 3,345

olist_orders_dataset.csv
order_id                       Unique: 99,441 | Duplicates: 0
customer_id                    Unique: 99,441 | Duplicates: 0

olist_products_dataset.csv
product_id                     Unique: 32,951 | Duplicates: 0

olist_sellers_dataset.csv
seller_id                      Unique: 3,095 | Duplicates: 0

olist_order_items_dataset.csv
order_id                       Unique: 98,666 | Duplicates: 13,984
order_item_id                  Unique: 21 | Duplicates: 112,629
product_id                     Unique: 32,951 | Duplicates: 79,699
seller_id                      Unique: 3,095 | Duplicates: 109,555

olist_order_payments_dataset.csv
order_id                       Unique: 99,440 | Duplicates: 4,446
payment_sequential             Unique: 29 | Duplicates: 103,857

olist_order_reviews_dataset.csv
review_id             

As part of the data profiling and modeling process, we need to identify which fields are expected to be unique and which can contain duplicate values.

- order_id in Orders: Expected to be unique, because each row represents one order.
- order_id in Order Items: Expected to be non-unique, because an order can contain multiple products.
- product_id in Products: Expected to be unique, because each row represents one product.
- product_id in Order Items: Expected to be non-unique, because the same product can be purchased across multiple orders.

# Let's understand the grain of each table 
# Orders

- order_id is unique: Each order is represented by exactly one row, confirming the expected grain of the table: 1 row = 1 order.
- customer_id is also unique in this particular Olist dataset: Each customer record is associated with a single order record in the Orders table.

# Order Items

The Order Items table has a different grain from the Orders table. Each row represents a product line within an order, meaning a single order can have multiple rows.
- order_id: 98,666 unique values across 112,650 rows.
This confirms that an order can contain multiple product items.
Therefore, order_id is not unique in the Order Items table.
- order_item_id: Contains only 21 unique values.
This may initially appear unusual, but it is expected.
order_item_id represents the line-item sequence number within an order (e.g., 1, 2, 3...), rather than a globally unique identifier.
Therefore, neither order_id nor order_item_id should be treated as a unique key for the Order Items table. The combination of order_id + order_item_id represents the appropriate unique line-item identifier.

# customers

The Customers table contains 99,441 customer records, but only 96,096 unique customer_unique_id values.

This indicates that some customers appear across multiple customer records in the dataset.

Specifically:

- Customer records: 99,441
- Unique customers (customer_unique_id): 96,096
- Difference: 3,345 records

The presence of repeated customer_unique_id values provides an opportunity to analyze customer purchasing behavior and retention.

Using customer_unique_id, we can derive meaningful business metrics such as:

- New vs. Returning Customers
- Number of Repeat Customers
- Customer Repeat Rate
- Customer Purchase Frequency

# Payments

he Payments table contains 103,886 payment records, compared with 99,440 unique orders.

This indicates that some orders have multiple payment records associated with them.

For example, a single order may have more than one payment entry due to:

- Multiple payment methods
- Split payments
- Installment-based payments

Therefore:

order_id is not necessarily unique in the Payments table.
The Payments table should be treated as a transaction-level table, where one order can have multiple payment records.

This distinction is important when calculating metrics such as total payment value, payment counts, and order-level revenue, as directly joining Payments to other line-item tables can result in double-counting.

For order-level analysis, payment records should be aggregated to the order grain before being joined where appropriate.

# Reviews

The Reviews table contains 99,224 review records, with 98,410 unique review_id values.

This indicates that there are 814 duplicate review IDs that require further investigation during the data-quality analysis.

More importantly, the order_id column contains 98,673 unique values, which indicates that reviews do not have a perfectly one-to-one relationship with orders.

Therefore, we should not assume that every order has exactly one review.

The relationship between Orders and Reviews will be determined based on the actual data rather than assumed business rules. This validation is important to ensure that review-related metrics, such as average review score and review count, are calculated accurately without introducing duplicate records or inflated results.

# 7. Missing values


In [ ]:
#python3 - <<'PY'
import pandas as pd
from pathlib import Path

data_path = Path("data/raw")   
for file in sorted(data_path.glob("*.csv")):
    df = pd.read_csv(file)

    missing = df.isna().sum()
    missing = missing[missing > 0]

    print("\n" + "=" * 70)
    print(file.name)

    if len(missing) == 0:
        print("No missing values")
    else:
        for column, count in missing.items():
            pct = count / len(df) * 100
            print(f"{column:40} {count:>8,} ({pct:6.2f}%)")
#PY


olist_customers_dataset.csv
No missing values

olist_geolocation_dataset.csv
No missing values

olist_order_items_dataset.csv
No missing values

olist_order_payments_dataset.csv
No missing values

olist_order_reviews_dataset.csv
review_comment_title                       87,656 ( 88.34%)
review_comment_message                     58,247 ( 58.70%)

olist_orders_dataset.csv
order_approved_at                             160 (  0.16%)
order_delivered_carrier_date                1,783 (  1.79%)
order_delivered_customer_date               2,965 (  2.98%)

olist_products_dataset.csv
product_category_name                         610 (  1.85%)
product_name_lenght                           610 (  1.85%)
product_description_lenght                    610 (  1.85%)
product_photos_qty                            610 (  1.85%)
product_weight_g                                2 (  0.01%)
product_length_cm                               2 (  0.01%)
product_height_cm                               2 (  0.0

# Reviews — mostly missing comments
This is not necessarily bad data.

A customer giving a rating of 5 but leaving the comment blank is perfectly valid.

So we should not replace these with fake text or delete those records.

# Orders — missing delivery dates
Missing delivery dates in the Orders table should not be replaced with averages, estimates, or other assumed values.

A NULL delivery date can represent a meaningful business condition. For example:

order_status = 'shipped'
order_delivered_customer_date = NULL

In this case, the NULL indicates that there is no recorded customer delivery date for the order. The order may still be in transit or may not have a completed delivery record.

Therefore, these missing values should be preserved as NULL in the data model rather than imputed with estimated or calculated dates.

This approach maintains the integrity of the source data and allows us to distinguish between actual delivery dates and orders without a recorded delivery date.


# Products — 610 records with missing category information

During data profiling, we observed that approximately 610 product records are missing several product metadata fields. The fact that these fields are missing together strongly suggests that the product metadata itself is incomplete for these records.

However, missing physical attributes such as product weight and dimensions are much less common, with only 2 products missing these values.

These records should be retained rather than removed.

For example:

product_id = X
product_category = NULL
product_weight = NULL

A missing product attribute does not mean that the product itself is invalid. The product may still appear in the Order Items table and represent legitimate sales activity.

Therefore, removing the product could result in the loss of valid transactional data.

# Important Data-Quality Principle

- Rule 1 — Don't delete missing review comments 
           Keep the review record.

- Rule 2 — Don't fabricate delivery dates
           Keep NULL.

- Rule 3 — Don't delete products with missing category
           Use Unknown/Unclassified in reporting.

- Rule 4 — Don't delete the 2 products missing dimensions
           Keep them.

- Rule 5 — Investigate before transforming


# 8. Referential Integrity Checks


In [10]:
# Referential integrity checks

checks = {}

checks["Order → Customer"] = (
    ~orders["customer_id"].isin(customers["customer_id"])
).sum()

checks["Order Item → Order"] = (
    ~order_items["order_id"].isin(orders["order_id"])
).sum()

checks["Order Item → Product"] = (
    ~order_items["product_id"].isin(products["product_id"])
).sum()

checks["Order Item → Seller"] = (
    ~order_items["seller_id"].isin(sellers["seller_id"])
).sum()

checks["Payment → Order"] = (
    ~payments["order_id"].isin(orders["order_id"])
).sum()

checks["Review → Order"] = (
    ~reviews["order_id"].isin(orders["order_id"])
).sum()

for relationship, orphan_count in checks.items():
    print(f"{relationship}: {orphan_count:,} orphan records")

Order → Customer: 0 orphan records
Order Item → Order: 0 orphan records
Order Item → Product: 0 orphan records
Order Item → Seller: 0 orphan records
Payment → Order: 0 orphan records
Review → Order: 0 orphan records


We're basically checking:


| Relationship         | Orphans | Meaning                                         |
| -------------------- | ------: | ----------------------------------------------- |
| Order → Customer     |       0 | Every order has a valid customer                |
| Order Item → Order   |       0 | Every item belongs to a valid order             |
| Order Item → Product |       0 | Every sold product exists in the product master |
| Order Item → Seller  |       0 | Every item has a valid seller                   |
| Payment → Order      |       0 | Every payment belongs to a valid order          |
| Review → Order       |       0 | Every review references a valid order           |


# 9. Date and Order Lifecycle Analysis

1. Date quality

For example:

- What date range do the orders cover?
- Are there orders where delivery happened before purchase?
- Are there orders delivered after the estimated date?
- Are there future dates?
- How many orders are still undelivered?

2. Business-rule validation

For example:

Can an order have a payment value of zero?

Can price be negative?

Can freight value be negative?

Can review score be outside 1–5?

Can quantity/order item numbers behave unexpectedly?

In [12]:
# Convert order date columns to datetime

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

# Display date range for each field

for col in date_columns:
    print(f"\n{col}")
    print("Earliest:", orders[col].min())
    print("Latest:  ", orders[col].max())
    print("Missing: ", orders[col].isna().sum())


order_purchase_timestamp
Earliest: 2016-09-04 21:15:19
Latest:   2018-10-17 17:30:18
Missing:  0

order_approved_at
Earliest: 2016-09-15 12:16:38
Latest:   2018-09-03 17:40:06
Missing:  160

order_delivered_carrier_date
Earliest: 2016-10-08 10:34:01
Latest:   2018-09-11 19:48:28
Missing:  1783

order_delivered_customer_date
Earliest: 2016-10-11 13:46:32
Latest:   2018-10-17 13:22:46
Missing:  2965

order_estimated_delivery_date
Earliest: 2016-09-30 00:00:00
Latest:   2018-11-12 00:00:00
Missing:  0


##  Order Lifecycle Metrics


In [13]:
# Order status distribution

orders["order_status"].value_counts(dropna=False)

delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: order_status, dtype: int64

In [14]:
# Delivery metrics for orders that have been delivered

delivered_orders = orders[
    orders["order_delivered_customer_date"].notna()
].copy()

delivered_orders["delivery_days"] = (
    delivered_orders["order_delivered_customer_date"]
    - delivered_orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

delivered_orders["days_vs_estimate"] = (
    delivered_orders["order_delivered_customer_date"]
    - delivered_orders["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60)

print("Delivered orders:", len(delivered_orders))
print("Average delivery days:", delivered_orders["delivery_days"].mean())
print("Median delivery days:", delivered_orders["delivery_days"].median())
print("Average days vs estimated:", delivered_orders["days_vs_estimate"].mean())

Delivered orders: 96476
Average delivery days: 12.558702304031792
Median delivery days: 10.21775462962963
Average days vs estimated: -11.179119879819492


In [15]:
# Late delivery rate

late_orders = (
    delivered_orders["order_delivered_customer_date"]
    > delivered_orders["order_estimated_delivery_date"]
)

print("Late delivered orders:", late_orders.sum())
print("Late delivery rate:", late_orders.mean() * 100, "%")

Late delivered orders: 7827
Late delivery rate: 8.112898544715783 %


Delivered status: 96,478

But Orders with a non-null delivery date: 96,476

So there are 2 delivered orders without a customer delivery timestamp.


# Let's make this as a business rule:

Delivery performance metrics are calculated only for orders with a valid customer delivery timestamp.

We can have meaningful business KPI:

Late delivery rate = 8.11%

That means roughly 8 out of every 100 delivered orders were delivered after the estimated date.

But there's an important distinction:

8.11% is not the same as “8.11% of all orders”

Our denominator is:

96,476 orders with an actual customer delivery date

So our definition is:

Late Delivery Rate = Late delivered orders ÷ Orders with valid delivery date

= 7,827 ÷ 96,476

= 8.11%

| KPI                            |               Result |
| ------------------------------ | -------------------: |
| Total orders                   |               99,441 |
| Delivered orders               |               96,478 |
| Orders with delivery date      |               96,476 |
| Average delivery time          |           12.56 days |
| Median delivery time           |           10.22 days |
| Late delivered orders          |                7,827 |
| **Late delivery rate**         |            **8.11%** |
| Average difference vs estimate | **11.18 days early** |


# let's see where the late deliveries are happening.
For example:

Is the 8.11% late rate concentrated in certain states?

In [17]:
# Add customer state to delivered orders

delivery_analysis = delivered_orders.merge(
    customers[["customer_id", "customer_state"]],
    on="customer_id",
    how="left"
)

state_delivery = (
    delivery_analysis
    .groupby("customer_state")
    .agg(
        delivered_orders=("order_id", "count"),
        late_orders=("days_vs_estimate", lambda x: (x > 0).sum()),
        avg_delivery_days=("delivery_days", "mean")
    )
    .reset_index()
)

state_delivery["late_delivery_rate"] = (
    state_delivery["late_orders"]
    / state_delivery["delivered_orders"]
    * 100
)

state_delivery.sort_values(
    "late_delivery_rate",
    ascending=False
).head(10)

,customer_state,delivered_orders,late_orders,avg_delivery_days,late_delivery_rate
1,AL,397,95,24.543855,23.929471
9,MA,717,141,21.572976,19.665272
16,PI,476,76,19.457098,15.966387
5,CE,1279,196,21.266579,15.324472
24,SE,335,51,21.519788,15.223881
4,BA,3256,457,19.335466,14.035627
18,RJ,12353,1664,15.310053,13.470412
26,TO,274,35,17.658063,12.773723
13,PA,946,117,23.772917,12.367865
7,ES,1995,244,15.789307,12.230576


Certain states exhibit substantially longer delivery times and higher late-delivery rates, warranting further investigation into geographic and logistics factors.